In [1]:
# ============================================================
# Notebook : 08_robustness.ipynb
# Purpose  : Two robustness analyses requested to strengthen the revision.
#   (A) Reproduction stability vs. sample size. The household set is split
#       into two disjoint groups of size k and the category's top lag-1
#       precursor is compared across groups, over many random splits and
#       several k. This shows (i) that at the study size all categories
#       reproduce, and (ii) that as the sample shrinks toward pilot scale
#       the weaker categories drop to PARTIAL reproduction - at which point
#       the two-dimensional HITL grid downgrades them from grade 1 to grade
#       2, a downgrade the precision-only rule would not make. This is the
#       empirical demonstration that the grid's correction actually bites.
#   (B) Bypass rate vs. causal-path width. The unconstrained-DiCE off-path
#       edit rate is recomputed when the causal path is widened from the top
#       precursor to the top-2 and top-3 precursors, to check that the high
#       bypass rate is not merely an artifact of a narrow (single-variable)
#       path definition.
# Design   : Single code cell. Grayscale figures, 600 dpi, png+pdf. No path print.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
import dice_ml

warnings.filterwarnings("ignore")
try:
    from functools import partialmethod
    import tqdm as _t, tqdm.auto as _ta
    _t.tqdm.__init__ = partialmethod(_t.tqdm.__init__, disable=True)
    _ta.tqdm.__init__ = partialmethod(_ta.tqdm.__init__, disable=True)
except Exception:
    pass

import random
SEED = 20260821
np.random.seed(SEED); random.seed(SEED)

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.unicode_minus": False,
                     "savefig.dpi": 600, "figure.dpi": 120, "axes.edgecolor": "0.15", "grid.color": "0.85"})
GRAY = ["0.10", "0.35", "0.55", "0.75"]
MARK = ["o", "s", "^", "D"]


def save_fig(fig, name):
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"
df = pd.read_pickle(DATA / "analysis_home.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")
perf = pd.read_csv(TAB / "table_model_performance.csv").set_index("category")
S = meta["sensor_cont"]
ABN = ["HR", "BR", "ST", "SP"]
CAT_LABEL = {"HR": "Heart rate", "BR": "Breath rate", "ST": "Skin temp.", "SP": "SpO2"}
elig = df[df["pcmci_eligible"] == 1]


def partial_corrs(y, X):
    n = len(y)
    Xd = np.column_stack([np.ones(n), X])
    beta, _, _, _ = np.linalg.lstsq(Xd, y, rcond=None)
    resid = y - Xd @ beta
    dof = n - Xd.shape[1]
    s2 = (resid @ resid) / dof
    inv = np.linalg.pinv(Xd.T @ Xd)
    se = np.sqrt(np.maximum(np.diag(inv) * s2, 1e-18))
    tv = beta / se
    return (tv / np.sqrt(tv ** 2 + dof))[1:]


# per-household precursor dicts
HH = {}
for hh, g in elig.groupby("HH_ID"):
    g = g.sort_values("Timestamp")
    if len(g) < 144:
        continue
    X = g[S].astype(float).values
    keep = X.std(0) > 1e-8
    names = [n for n, k in zip(S, keep) if k]
    Xl = X[:-1][:, keep]; Xl = (Xl - Xl.mean(0)) / Xl.std(0)
    d = {}
    for c in ABN:
        yv = g["cat_" + c].astype(float).values; y, yl = yv[1:], yv[:-1]
        if y.std() < 1e-8 or yl.std() < 1e-8:
            continue
        v = partial_corrs(y, np.column_stack([Xl, (yl - yl.mean()) / yl.std()]))[:-1]
        d[c] = {nm: float(x) for nm, x in zip(names, v)}
    HH[hh] = d
used = list(HH.keys())


def agg(hhs):
    acc = {c: {} for c in ABN}
    for hh in hhs:
        for c, dd in HH[hh].items():
            for s, x in dd.items():
                acc[c].setdefault(s, []).append(x)
    return {c: {s: float(np.mean(v)) for s, v in dd.items()} for c, dd in acc.items()}


def top(d):
    return max(d, key=lambda k: abs(d[k])) if d else None


# ============================================================
# (A) Reproduction stability vs sample size k (per group).
# ============================================================
KS = [3, 4, 5, 6, 8, 11]
NSPLIT = 300
stab = {c: [] for c in ABN}
for k in KS:
    cnt = {c: [0, 0] for c in ABN}
    for _ in range(NSPLIT):
        perm = used[:]; random.shuffle(perm)
        A, B = perm[:k], perm[k:2 * k]
        pa, pb = agg(A), agg(B)
        for c in ABN:
            if pa[c] and pb[c]:
                cnt[c][1] += 1
                if top(pa[c]) == top(pb[c]):
                    cnt[c][0] += 1
    for c in ABN:
        stab[c].append(cnt[c][0] / cnt[c][1] if cnt[c][1] else np.nan)
stab_tbl = pd.DataFrame(stab, index=[f"k={k}" for k in KS])
stab_tbl.to_csv(TAB / "table_reproduction_vs_k.csv", encoding="utf-8-sig")

# ---- Pilot-scale HITL demonstration (k = 3 per group, N = 6) ----
def verdict(top1):
    if top1 >= 0.9:
        return "full"
    if top1 >= 0.5:
        return "partial"
    return "not_reproduced"


def grade2d(v, p):
    if pd.isna(p) or p < 0.10:
        return 3
    if v == "full" and p >= 0.40:
        return 1
    return 2


def grade1d(p):
    if pd.isna(p) or p < 0.10:
        return 3
    return 1 if p >= 0.40 else 2


k_pilot = KS.index(3)
rows = []
for c in ABN:
    p = float(perf.loc[c, "precision"])
    v_full = "full"                                   # at study size all reproduce
    v_pilot = verdict(stab[c][k_pilot])
    rows.append({"category": CAT_LABEL[c], "precision": round(p, 3),
                 "repro_full_sample": v_full, "grade_full_sample": grade2d(v_full, p),
                 "top1_agree_pilot(k=3)": round(stab[c][k_pilot], 3),
                 "repro_pilot": v_pilot, "grade_pilot_2D": grade2d(v_pilot, p),
                 "grade_pilot_precision_only": grade1d(p),
                 "downgraded_by_grid_at_pilot": grade2d(v_pilot, p) > grade1d(p)})
hitl_demo = pd.DataFrame(rows)
hitl_demo.to_csv(TAB / "table_hitl_pilot_demo.csv", index=False, encoding="utf-8-sig")

# ---- Figure: reproduction stability vs k ----
fig, ax = plt.subplots(figsize=(5.6, 3.6))
xs = [k for k in KS]
for i, c in enumerate(ABN):
    ax.plot(xs, stab[c], marker=MARK[i], color=GRAY[i], linewidth=1.3,
            markersize=5, markerfacecolor="white", markeredgecolor=GRAY[i], label=CAT_LABEL[c])
ax.axhline(0.9, color="black", linestyle="--", linewidth=0.9)
ax.set_xlabel("Households per group (k)")
ax.set_ylabel("Top-precursor agreement across halves")
ax.set_ylim(0.6, 1.02)
ax.legend(frameon=False, fontsize=8)
save_fig(fig, "fig_reproduction_vs_k")

# ============================================================
# (B) Bypass rate vs causal-path width (top-1 / top-2 / top-3 precursors).
# ============================================================
full_prec = agg(used)
rank = {c: [s for s, _ in sorted(((s, abs(v)) for s, v in full_prec[c].items()),
                                 key=lambda t: -t[1])] for c in ABN}
sd = df[S].std().replace(0, 1)


def train(cat):
    y = df["cat_" + cat].values
    X = df[S].astype(float)
    gr = df["HH_ID"].values
    tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=SEED).split(X, y, gr))
    clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced",
                                 random_state=SEED, n_jobs=-1).fit(X.iloc[tr], y[tr])
    tr_df = X.iloc[tr].copy(); tr_df["target"] = y[tr]
    te_df = X.iloc[te].copy(); te_df["target"] = y[te]
    return clf, tr_df, te_df


brows = []
for cat in ABN:
    clf, tr_df, te_df = train(cat)
    d = dice_ml.Data(dataframe=tr_df, continuous_features=S, outcome_name="target")
    m = dice_ml.Model(model=clf, backend="sklearn")
    exp = dice_ml.Dice(d, m, method="random")
    pos = te_df[te_df["target"] == 1].drop(columns=["target"])
    if len(pos) == 0:
        continue
    q = pos.sample(min(40, len(pos)), random_state=SEED).reset_index(drop=True)
    tops = []
    for i in range(len(q)):
        base = q.iloc[i]
        try:
            cf = exp.generate_counterfactuals(q.iloc[[i]], total_CFs=5, desired_class="opposite",
                                              features_to_vary="all", verbose=False)
            f = cf.cf_examples_list[0].final_cfs_df
        except Exception:
            f = None
        if f is None:
            continue
        for _, r in f.iterrows():
            ch = {v: abs((r[v] - base[v]) / sd[v]) for v in S}
            tops.append(max(ch, key=ch.get))
    n = len(tops)
    brows.append({"category": CAT_LABEL[cat], "n_cf": n,
                  "offpath_top1": round(float(np.mean([t not in set(rank[cat][:1]) for t in tops])), 3),
                  "offpath_top2": round(float(np.mean([t not in set(rank[cat][:2]) for t in tops])), 3),
                  "offpath_top3": round(float(np.mean([t not in set(rank[cat][:3]) for t in tops])), 3),
                  "top3_path": ", ".join(rank[cat][:3])})
bw = pd.DataFrame(brows)
bw.to_csv(TAB / "table_bypass_width.csv", index=False, encoding="utf-8-sig")

# ---- Figure: bypass rate vs path width ----
fig, ax = plt.subplots(figsize=(5.8, 3.4))
x = np.arange(len(bw)); w = 0.26
for j, (col, lab, sh) in enumerate([("offpath_top1", "top-1 path", "0.25"),
                                    ("offpath_top2", "top-2 path", "0.55"),
                                    ("offpath_top3", "top-3 path", "0.8")]):
    ax.bar(x + (j - 1) * w, bw[col], w, color=sh, edgecolor="black", linewidth=0.5, label=lab)
ax.set_xticks(x); ax.set_xticklabels(bw["category"])
ax.set_ylabel("Off-path edit rate")
ax.set_ylim(0, 1.0)
ax.legend(frameon=False, fontsize=8)
save_fig(fig, "fig_bypass_width")

# ---- Console summary ----
print("=== (A) Reproduction stability vs sample size ===")
print(stab_tbl.round(3).to_string())
print()
print("Pilot-scale HITL demonstration (k=3 per group, N=6):")
print(hitl_demo.to_string(index=False))
n_down = int(hitl_demo["downgraded_by_grid_at_pilot"].sum())
print(f"-> at pilot scale the 2D grid downgrades {n_down} categories that precision-only would grade 1.")
print()
print("=== (B) Bypass rate vs causal-path width ===")
print(bw.to_string(index=False))
print(f"mean off-path: top1={bw.offpath_top1.mean():.3f}, top2={bw.offpath_top2.mean():.3f}, top3={bw.offpath_top3.mean():.3f}")
print()
print("Figures: fig_reproduction_vs_k, fig_bypass_width (png+pdf, 600 dpi)")
print("Tables : table_reproduction_vs_k, table_hitl_pilot_demo, table_bypass_width")


=== (A) Reproduction stability vs sample size ===
         HR     BR     ST     SP
k=3   0.840  0.757  0.993  0.793
k=4   0.940  0.870  1.000  0.880
k=5   0.983  0.900  1.000  0.937
k=6   0.993  0.957  1.000  0.953
k=8   1.000  0.990  1.000  0.997
k=11  1.000  1.000  1.000  1.000

Pilot-scale HITL demonstration (k=3 per group, N=6):
   category  precision repro_full_sample  grade_full_sample  top1_agree_pilot(k=3) repro_pilot  grade_pilot_2D  grade_pilot_precision_only  downgraded_by_grid_at_pilot
 Heart rate      0.760              full                  1                  0.840     partial               2                           1                         True
Breath rate      0.637              full                  1                  0.757     partial               2                           1                         True
 Skin temp.      0.662              full                  1                  0.993        full               1                           1                       